# Day 10 · Prompting only — The Line Printer

**Question.** At nine at night, how long until a rider who cannot take the stairs is moving: rolled
to a bus or to a train at an accessible station, and picked up?

**Approach.** Print the city the way the first computer maps were printed. SYMAP, begun after a
1963 course at Northwestern, built tone on a line printer by striking characters over each other.
Here one character is one city block, an eighth of a mile on Chicago's address grid. For every
block: the step-free walk at 3 ft/s to each CTA bus route and each accessible 'L' station within
four blocks, plus half the time between its buses or trains at 21:00. The block prints the least.

Made from start to finish in one command-line session with an AI assistant (the theme); the
prompts are published beside this notebook.

| Section | What happens |
| --- | --- |
| 1. Setup | Imports, paths, settings |
| 2. Pull | CTA GTFS, accessible stations, blocks, city limit, step-free walk network |
| 3. Validate | Feed covers the day; departures in the window; residents |
| 4. The printer's grid | One cell per block inside the city; residents per cell |
| 5. Walk + wait | Per route: step-free walk to its nearest stop, half its headway |
| 6. Analyze | Classes, the nine sides, the inset, the finding |
| 7. QA view | Quick map and checks |
| 8. Export | `plate.json` |

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import logging
import pickle
import re
import zipfile
from datetime import date
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely
from scipy.spatial import cKDTree

from atlas import Ledger, write_plate
from atlas import grid, walk

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
log = logging.getLogger('day10')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)

In [ ]:
# Run settings ---------------------------------------------------------------------------------
DAY = 10
SERVICE_DAY = date(2026, 11, 10)  # Tuesday 10 November 2026, the challenge's Day 10
DOW = 'tuesday'
WINDOW = (20.5, 21.5)             # departures 20:30-21:30, centred on 21:00
WALK_M = walk.HALF_MILE_M         # four blocks
SPEED = 0.9144                    # m/s = 3.0 ft/s: MUTCD (2009) 4E.06 times a whole crossing at 3 ft/s (para 14), vs 3.5 ft/s for clearance
CELL = 1 / 8                      # miles: one block
MAX_STREET_M = 100                # a cell prints only if a walkway is this close to its centre
BREAKS = [7.5, 10, 12.5, 15, 20]  # minutes; classes 0-5, 6 = nothing within four blocks
CLASS_LABELS = ['under 7½ min', '7½–10', '10–12½', '12½–15', '15–20', 'over 20', 'nothing within 4 blocks']
EXTENT = (-17, 6, -17, 16)        # miles west, east, south, north (13600 W-4800 E, 14000 S-12800 N)
INSET = dict(west=-3, east=0.5, south=-14, north=-10.5)  # Beverly to Roseland (3200 W-400 E, 8800 S-11600 S), miles

DAY_DIR = Path.cwd()
REPO = DAY_DIR.parents[1]
assert (DAY_DIR / 'day10.ipynb').exists(), f'Run this notebook from its own folder, not {DAY_DIR}'
GTFS_DIR = REPO / 'reference' / 'raw' / 'gtfs'
OSM_DIR = REPO / walk.OSM_DIR
BLOCKS_CSV = REPO / 'reference' / 'chicago-blocks-2020.csv'
BOUNDARY_RAW = sorted((REPO / 'reference' / 'raw').glob('city_boundary_*.geojson'))[-1]
STATIONS_JSON = REPO / 'days' / 'day08-utopia' / 'output' / 'plate.json'  # ADA flags with Racine, Austin fixed
PROC_DIR = DAY_DIR / 'data' / 'processed'
OUT_PATH = DAY_DIR / 'output' / 'plate.json'
GTFS_URL = 'https://www.transitchicago.com/developers/gtfs/'
MUTCD_URL = 'https://mutcd.fhwa.dot.gov/htm/2009/part4/part4e.htm'
RAIL = {'Red': 'Red', 'Blue': 'Blue', 'Brn': 'Brown', 'G': 'Green', 'Org': 'Orange', 'P': 'Purple', 'Pink': 'Pink', 'Y': 'Yellow'}

# Chicago's nine sides, by community area: the grouping of Wikipedia's 'Community areas in Chicago' (unofficial)
SIDES_URL = 'https://en.wikipedia.org/wiki/Community_areas_in_Chicago'
SIDES = {
    'Central': [8, 32, 33],
    'North Side': [5, 6, 7, 21, 22],
    'Far North Side': [1, 2, 3, 4, 9, 10, 11, 12, 13, 14, 76, 77],
    'Northwest Side': [15, 16, 17, 18, 19, 20],
    'West Side': [23, 24, 25, 26, 27, 28, 29, 30, 31],
    'South Side': [34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 60, 68, 69],
    'Southwest Side': [56, 57, 58, 59, 61, 62, 63, 64, 65, 66, 67],
    'Far Southwest Side': [70, 71, 72, 73, 74, 75],
    'Far Southeast Side': [44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55],
}
SIDE_OF = {ca: s for s, cas in SIDES.items() for ca in cas}
assert sorted(SIDE_OF) == list(range(1, 78))

## 2. Pull

Everything here is already on disk from earlier days: the CTA GTFS snapshot (gitignored, dated),
the 2020 blocks with their community areas, the city limit, the Illinois OSM extract and its
cached walk graph. Accessible stations come from Day 08's table, which corrects the portal's ADA
flag for Racine (Blue, reopened Oct 2025) and Austin (Green, Aug 2026).

In [ ]:
gtfs_path = sorted(GTFS_DIR.glob('cta_gtfs_*.zip'))[-1]
gtfs_date = date.fromisoformat(re.search(r'(\d{4}-\d{2}-\d{2})', gtfs_path.name).group(1))
D = int(SERVICE_DAY.strftime('%Y%m%d'))


def read(z: zipfile.ZipFile, name: str, cols: set[str] | None = None) -> pd.DataFrame:
    with z.open(name) as f:
        return pd.read_csv(f, dtype=str, usecols=lambda c: cols is None or c in cols)


with zipfile.ZipFile(gtfs_path) as z:
    cal = read(z, 'calendar.txt')
    on = set(cal[(cal[DOW] == '1') & (cal['start_date'].astype(int) <= D) & (cal['end_date'].astype(int) >= D)]['service_id'])
    if 'calendar_dates.txt' in z.namelist():
        cd = read(z, 'calendar_dates.txt')
        cd = cd[cd['date'].astype(int) == D]
        on |= set(cd.loc[cd['exception_type'] == '1', 'service_id'])
        on -= set(cd.loc[cd['exception_type'] == '2', 'service_id'])
    routes = read(z, 'routes.txt', {'route_id', 'route_type', 'route_short_name', 'route_long_name'})
    trips = read(z, 'trips.txt', {'trip_id', 'service_id', 'route_id', 'direction_id'})
    trips = trips[trips['service_id'].isin(on)].merge(routes, on='route_id')
    st = read(z, 'stop_times.txt', {'trip_id', 'departure_time', 'stop_id'})
    gstops = read(z, 'stops.txt', {'stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'parent_station'})
st = st[st['trip_id'].isin(set(trips['trip_id']))]
hms = st['departure_time'].str.split(':', expand=True).astype(int)
st = st.assign(h=hms[0] + hms[1] / 60 + hms[2] / 3600)
log.info('%s: %d services, %d trips, %d stop departures on %s', gtfs_path.name, len(on), len(trips), len(st), SERVICE_DAY)

doc8 = json.loads(STATIONS_JSON.read_text())
stations = pd.DataFrame([s for s in doc8['stations'] if s['id'].isdigit()])
ADA = dict(zip(stations['id'], stations['ada']))
blocks = pd.read_csv(BLOCKS_CSV, dtype={'geoid': str})
city_ll = shapely.union_all(gpd.read_file(BOUNDARY_RAW).geometry)
boundary_pulled = BOUNDARY_RAW.stem.rsplit('_', 1)[-1]
P = int(blocks['pop'].sum())
log.info('%d stations (%d accessible), %d blocks, %d residents', len(stations), stations['ada'].sum(), len(blocks), P)

In [ ]:
# Step-free walk network (Day 03 rules), cached
pbf, osm_date = walk.latest_pbf(OSM_DIR)
cache = PROC_DIR / f'stepfree-edges-{osm_date.isoformat()}.pkl'
if cache.exists():
    E = pickle.loads(cache.read_bytes())
else:
    xml = OSM_DIR / f'chicago-walk-{osm_date.isoformat()}.osm'
    assert xml.exists(), 'build the walk extract first (Day 03 notebook)'
    E = walk.Edges.of(walk.main_component(walk.step_free(walk.load(xml))))
    PROC_DIR.mkdir(parents=True, exist_ok=True)
    cache.write_bytes(pickle.dumps(E))
log.info('Step-free network: %d edges (OSM %s)', len(E.length), osm_date)

## 3. Validate

In [ ]:
assert on, f'no CTA service on {SERVICE_DAY}'
win = st[(st['h'] >= WINDOW[0]) & (st['h'] < WINDOW[1])].merge(
    trips[['trip_id', 'route_id', 'route_type', 'direction_id']], on='trip_id').merge(gstops, on='stop_id')
rail = win['route_type'].eq('1')
win['station'] = np.where(rail, win['parent_station'], '')
unknown = set(win.loc[rail, 'station']) - set(ADA)
assert not unknown, f'rail stations missing from the Day 08 table: {unknown}'
win = win[~rail | win['station'].map(ADA).astype(bool)]   # trains only at accessible stations
n_routes = win.groupby('route_type')['route_id'].nunique().to_dict()
log.info('In the window: %d departures, routes by type %s', len(win), n_routes)
assert n_routes.get('3', 0) > 80 and n_routes.get('1', 0) >= 7
assert 2_700_000 < P < 2_800_000

## 4. The printer's grid

One cell per block: an eighth of a mile on the address grid, measured from State and Madison. A
cell prints when its centre is inside the city and a walkway is within 100 m of it (parks, the
airports, rail yards and the river print blank). Residents are placed by their block's internal
point.

In [ ]:
def ll_to_miles(g):
    return shapely.transform(g, lambda c: np.column_stack(grid.miles_from_origin(c[:, 1], c[:, 0])))


city = ll_to_miles(city_ll)
x0, y0, x1, y1 = city.bounds
cols = np.arange(np.floor(x0 / CELL), np.ceil(x1 / CELL)).astype(int)
rows = np.arange(np.floor(y0 / CELL), np.ceil(y1 / CELL)).astype(int)
cc, rr = np.meshgrid(cols, rows)
cells = pd.DataFrame({'col': cc.ravel(), 'row': rr.ravel()})
cells['e'] = (cells['col'] + 0.5) * CELL
cells['n'] = (cells['row'] + 0.5) * CELL
cells = cells[shapely.contains_xy(city, cells['e'], cells['n'])].reset_index(drop=True)
lat = grid.ORIGIN_LAT + cells['n'] / grid.MILES_PER_DEG_LAT
lon = grid.ORIGIN_LON + cells['e'] / grid.MILES_PER_DEG_LON
cxy = gpd.GeoSeries(gpd.points_from_xy(lon, lat), crs=4326).to_crs(walk.CRS).get_coordinates().to_numpy()
# every walkway within 100 m of a cell's centre is a way out of it (the single nearest can be a
# lakefront path or a rail-side walk whose step-free links are far away); cells with none take their nearest
tree = shapely.STRtree(E.geometry)
pts = shapely.points(cxy)
pi, ei = tree.query(pts, predicate='dwithin', distance=MAX_STREET_M)
lone = np.setdiff1d(np.arange(len(cells)), pi)
(_, near_e) = tree.query_nearest(pts[lone], all_matches=False)
pi, ei = np.r_[pi, lone], np.r_[ei, near_e]
along = shapely.line_locate_point(E.geometry[ei], pts[pi])
off = shapely.distance(E.geometry[ei], pts[pi])
cells['street_m'] = pd.Series(off).groupby(pi).min().reindex(range(len(cells))).to_numpy()
log.info('%d cell-walkway pairs', len(pi))

be, bn = grid.miles_from_origin(blocks['lat'].to_numpy(), blocks['lon'].to_numpy())
blocks['col'], blocks['row'] = np.floor(be / CELL).astype(int), np.floor(bn / CELL).astype(int)
key = cells.set_index(['col', 'row']).index
blocks['cell'] = key.get_indexer(pd.MultiIndex.from_arrays([blocks['col'], blocks['row']]))
# blocks whose cell centre is outside the city (edge blocks) go to the nearest cell
lost = blocks['cell'] < 0
_, near = cKDTree(cells[['e', 'n']].to_numpy()).query(np.c_[be[lost], bn[lost]])
blocks.loc[lost, 'cell'] = near
log.info('%d cells in the city; %d blocks re-homed from edge cells (%d residents)', len(cells), lost.sum(), blocks.loc[lost, 'pop'].sum())

## 5. Walk + wait

For each route in the window (buses, and trains at accessible stations only), one multi-source
walk from all its stops on the step-free network. A cell reaches a route when its nearest stop of
that route is within four blocks (half a mile) on foot. Its departures an hour at that stop (one
direction) give the headway; the expected wait is half of it, for a rider who arrives at random.
Walk time at 3 ft/s. The cell keeps the route with the least walk + wait.

In [ ]:
per = (win.groupby(['route_id', 'route_type', 'stop_id'])
       .agg(n=('h', 'size'), lat=('stop_lat', 'first'), lon=('stop_lon', 'first')).reset_index())
per['tph'] = per['n'] / (WINDOW[1] - WINDOW[0])
nc = len(cells)
best = dict(t=np.full(nc, np.inf), route=np.full(nc, '', dtype=object), walk=np.full(nc, np.inf), tph=np.zeros(nc))
for (route, rtype), g in per.groupby(['route_id', 'route_type']):
    sxy = gpd.GeoSeries(gpd.points_from_xy(g['lon'].astype(float), g['lat'].astype(float)), crs=4326).to_crs(walk.CRS).get_coordinates().to_numpy()
    dist, lab, _ = walk.reach(E, list(g['stop_id']), sxy, limit=WALK_M * 1.2)
    du = dist[E.u[ei]] + along
    dv = dist[E.v[ei]] + (E.length[ei] - along)
    dp = np.minimum(du, dv) + off
    kp = np.where(du <= dv, lab[E.u[ei]], lab[E.v[ei]])
    pair = pd.DataFrame({'cell': pi, 'd': dp, 'k': kp}).sort_values('d').drop_duplicates('cell').set_index('cell').reindex(range(nc))
    d, k = pair['d'].fillna(np.inf).to_numpy(), pair['k'].fillna('').to_numpy()
    tph = np.where(d <= WALK_M, pd.Series(k).map(g.set_index('stop_id')['tph']).fillna(0).to_numpy(), 0)
    t = np.where(tph > 0, d / SPEED / 60 + 30 / np.maximum(tph, 1e-9), np.inf)
    sooner = t < best['t']
    for name, v in (('t', t), ('walk', d), ('tph', tph)):
        best[name][sooner] = v[sooner]
    best['route'][sooner] = route
cells['t'], cells['route'], cells['walk_m'], cells['tph'] = best['t'], best['route'], best['walk'], best['tph']
cells['printed'] = cells['street_m'] <= MAX_STREET_M
cells['k'] = np.where(np.isinf(cells['t']), 6, np.searchsorted(BREAKS, cells['t'].to_numpy(), side='left'))
log.info('%d routes walked; %d cells printed', per['route_id'].nunique(), cells['printed'].sum())

## 6. Analyze

In [ ]:
b = blocks.join(cells[['t', 'k', 'route', 'printed']], on='cell')
# edge blocks with no community area (Day 03: 40 residents) take their nearest block's
no_ca = b['commarea'].eq(0)
_, nn = cKDTree(np.c_[be[~no_ca], bn[~no_ca]]).query(np.c_[be[no_ca], bn[no_ca]])
b.loc[no_ca, 'commarea'] = b.loc[~no_ca, 'commarea'].to_numpy()[nn]
b['side'] = b['commarea'].map(SIDE_OF)
assert b['side'].notna().all()
unprinted = int(b.loc[~b['printed'], 'pop'].sum())


def median_t(df):
    o = df.sort_values('t')
    cp = o['pop'].cumsum()
    return float(o['t'].iloc[int(np.searchsorted(cp, cp.iloc[-1] / 2))])


classes = b.groupby('k')['pop'].sum().reindex(range(7), fill_value=0)
amb = b.groupby('k')['amb'].sum().reindex(range(7), fill_value=0)
sides = (b.groupby('side').apply(lambda g: pd.Series({
    'pop': int(g['pop'].sum()), 'median': median_t(g),
    'over15': g.loc[g['t'] > 15, 'pop'].sum() / g['pop'].sum(),
    'mix': (g.groupby('k')['pop'].sum().reindex(range(7), fill_value=0) / g['pop'].sum()).round(4).tolist(),
}), include_groups=False).sort_values('median'))
TRAIN = set(RAIL)
summary = {
    'residents': P,
    'median': round(median_t(b), 1),
    'over15': int(b.loc[b['t'] > 15, 'pop'].sum()),
    'over15Amb': round(float(b.loc[b['t'] > 15, 'amb'].sum())),
    'none': int(classes[6]),
    'trainBest': int(b.loc[b['route'].isin(TRAIN), 'pop'].sum()),
    'classes': [int(v) for v in classes],
    'unprintedResidents': unprinted,
}
display(sides)
summary

In [ ]:
# The finding (every number from the data; one rounding per number)
worst, best_side = sides['over15'].idxmax(), sides['over15'].idxmin()
pct = lambda v: f'{100 * v:.0f}%'
FINDING = (f"At 9 p.m., half of Chicago's residents need more than {int(summary['median'])} minutes to get moving: "
           f"a step-free roll to a bus or an accessible train, and the wait for it. On the {worst}, "
           f"{pct(sides.loc[worst, 'over15'])} need more than 15; on the {best_side}, {pct(sides.loc[best_side, 'over15'])}.")
FINDING

In [ ]:
# The inset: every cell with its details (the day page shows them on hover)
# the day page draws the inset tall (7 miles above and below its centre) and crops it, so the band is exported
mid = (INSET['north'] + INSET['south']) / 2
ins = cells[(cells['e'] >= INSET['west']) & (cells['e'] <= INSET['east']) & (cells['n'] >= mid - 7) & (cells['n'] <= mid + 7)]
rname = {r.route_id: (f'{RAIL[r.route_id]} Line' if r.route_id in RAIL else f'#{r.route_short_name} {r.route_long_name}')
         for r in routes.itertuples()}
log.info('inset: %d cells, %d residents', len(ins), b.loc[b['cell'].isin(ins.index), 'pop'].sum())

## 7. QA view

In [ ]:
fig, ax = plt.subplots(figsize=(7, 9))
p = cells[cells['printed']]
ax.scatter(p['e'], p['n'], c=p['k'], s=1.2, marker='s', cmap='Greys', vmin=-1, vmax=6)
ax.set_aspect('equal'); ax.set_title('Minutes until moving, 21:00 (darker = longer)')
plt.show()
# cells with nothing within four blocks but a 21:00 stop close by in a straight line: walk-network gaps?
stops_xy = np.c_[grid.miles_from_origin(per['lat'].astype(float).to_numpy(), per['lon'].astype(float).to_numpy())]
dd, _ = cKDTree(stops_xy).query(cells[['e', 'n']].to_numpy())
cells['crow_mi'] = dd
gap = cells[cells['printed'] & (cells['k'] == 6) & (cells['crow_mi'] < 0.25)]
log.info('QA: %d printed cells unserved but within 0.25 mi of a 21:00 stop (%d residents)', len(gap), b.loc[b['cell'].isin(gap.index), 'pop'].sum())
gap.assign(pop=b.groupby('cell')['pop'].sum().reindex(gap.index).fillna(0)).sort_values('pop', ascending=False).head(10)

## 8. Export

In [ ]:
def addr(e, n):
    return round(float(e * 800)), round(float(grid.north_to_address(n)))


# rows as strings: '0'-'6' = class, ' ' = blank (outside the city or no street)
rows_out = []
for row, g in cells.groupby('row'):
    c0, c1 = int(g['col'].min()), int(g['col'].max())
    s = [' '] * (c1 - c0 + 1)
    for c, k, pr in zip(g['col'], g['k'], g['printed']):
        if pr:
            s[c - c0] = str(int(k))
    rows_out.append({'row': int(row), 'col': c0, 's': ''.join(s).rstrip()})

r1 = lambda v: None if not np.isfinite(v) else round(float(v), 1)
# inset cells, compact: [col, row, minutes, walk minutes, minutes between buses or trains, route index]
used = sorted(set(ins.loc[ins['printed'] & (ins['route'] != ''), 'route']))
route_names = [rname[r] for r in used]
ri = {r: i for i, r in enumerate(used)}
inset_cells = [[int(c.col), int(c.row), r1(c.t), r1(c.walk_m / SPEED / 60), r1(60 / c.tph) if c.tph > 0 else None, ri.get(c.route)]
               for c in ins[ins['printed']].itertuples()]
marks = [{'id': s['id'], 'name': s['name'], 'short': s['short'], 'lines': s['lines'], 'ada': bool(s['ada']), 'x': s['x'], 'y': s['y']}
         for s in doc8['stations'] if s['id'].isdigit()]

ledger = Ledger(
    source='CTA GTFS schedule; OpenStreetMap walk network; 2020 Census blocks',
    dataset_id='GTFS',
    url=GTFS_URL,
    pulled=gtfs_date,
    units=f'{int(cells["printed"].sum()):,} blocks of ⅛ mile on Chicago’s address grid; {P:,} residents (2020 Census)',
    method=[
        f'Weekday {SERVICE_DAY:%A %-d %B %Y}, departures 20:30–21:30; every CTA bus route, and trains at accessible stations only',
        f'Step-free walk (no stairs) on OpenStreetMap ({osm_date:%-d %b %Y}) from each block’s centre, by any walkway within 100 m, to each route’s nearest stop within four blocks (½ mile), at 3 ft/s (MUTCD 2009, 4E.06: the speed for timing a whole crossing; 3.5 ft/s for clearance)',
        'Minutes until moving = that walk + half the time between buses or trains at the stop; each block prints the least',
        'Printed as a line-printer map after SYMAP (1963–66): one character per block, tone by overprinting',
    ],
    caveats=[
        'Scheduled service, not real arrivals; one direction at the nearest stop; a rider arriving at random',
        'CTA: all buses and railcars accessible; bus stops themselves (curb, shelter, snow) are not checked',
        'Blocks with no walkway within 100 m print blank; residents placed by their 2020 block’s internal point',
    ],
)
payload = {
    'serviceDay': SERVICE_DAY.isoformat(),
    'window': list(WINDOW),
    'speedFtS': 3.0,
    'walkM': WALK_M,
    'cell': CELL,
    'breaks': BREAKS,
    'classLabels': CLASS_LABELS,
    'extent': dict(zip(['west', 'east', 'south', 'north'], EXTENT)),
    'insetBox': INSET,
    'gtfs': {'snapshot': gtfs_date.isoformat(), 'url': GTFS_URL, 'routes': int(per['route_id'].nunique())},
    'osm': osm_date.isoformat(),
    'sources': {'mutcd': MUTCD_URL, 'boundary': boundary_pulled, 'sides': SIDES_URL},
    'summary': summary,
    'sides': [{'side': s, 'pop': int(r['pop']), 'median': round(float(r['median']), 1), 'over15': round(float(r['over15']), 4), 'mix': r['mix']}
              for s, r in sides.iterrows()],
    'rows': rows_out,
    'inset': {'box': INSET, 'title': 'Beverly to Roseland', 'cells': inset_cells, 'routes': route_names},
    'stations': marks,
}
path = write_plate(OUT_PATH, day=DAY, ledger=ledger, finding=FINDING, payload=payload)
assert 'NaN' not in path.read_text(), 'NaN is not valid JSON for the site'
log.info('Wrote %s (%.0f KB)', path.relative_to(DAY_DIR), path.stat().st_size / 1024)